# Check GPU

In [1]:
!nvidia-smi

Thu Aug 20 09:27:54 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   46C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

# Connect to HuggingFace

In [1]:
from huggingface_hub import notebook_login
notebook_login()

# Install all dependencies

In [2]:
!pip install -q transformers peft bitsandbytes trl accelerate datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 925.8/925.8 kB 62.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 50.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 12.9 MB/s eta 0:00:00


# Mount to Drive to save progress

In [4]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os
SAVE_DIR = '/content/drive/MyDrive/qlora-sql-project'
os.makedirs(SAVE_DIR, exist_ok=True)

Mounted at /content/drive


# Import the model and quantize it so it fits inside the GPU

In [5]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_id = "mistralai/Mistral-7B-Instruct-v0.3"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
)

config.json:   0%|          | 0.00/601 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/141k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  587kB            

tokenizer.model: downloading bytes:           |  0.00B            

...

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

# Run Check

In [6]:
prompt = "What is 2+2? Answer in one word."
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
output = model.generate(**inputs, max_new_tokens=10)
print(tokenizer.decode(output[0], skip_special_tokens=True))

What is 2+2? Answer in one word.

Four.

What is the


# Load dataset, pick 3000 items after shuffle, and create a function for processing prompts

In [7]:
from datasets import load_dataset

dataset = load_dataset("b-mc2/sql-create-context", split="train")
dataset = dataset.shuffle(seed=42).select(range(3000))

def format_example(example):
    text = f"""[INST] Given this database schema:
{example['context']}

Write a SQL query for: {example['question']} [/INST] {example['answer']}"""
    return {"text": text}

dataset = dataset.map(format_example)

README.md:   0%|          | 0.00/4.43k [00:00<?, ?B/s]

sql_create_context_v4.json: reconstructing file:   0%|          |  0.00B / 21.8MB            

sql_create_context_v4.json: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/78577 [00:00<?, ? examples/s]

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

# Configure and Run the model, can take 45mins+, keep a check every 10-15 minutes

In [10]:
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

training_args = SFTConfig(
    output_dir=f"{SAVE_DIR}/checkpoints",
    num_train_epochs=1,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=10,
    save_steps=50,
    save_total_limit=2,
    bf16=True,
    dataset_text_field="text",
    max_length=512,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    peft_config=lora_config,
)

trainer.train()

Adding EOS to train dataset:   0%|          | 0/3000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/3000 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/3000 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/3000 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/3000 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Step,Training Loss
10,1.901251
20,1.142145
30,1.055019
40,0.962174
50,0.953219
60,0.927501
70,0.910051
80,0.887420
90,0.902550
100,0.863636


TrainOutput(global_step=188, training_loss=0.9570221038574868, metrics={'train_runtime': 6267.9505, 'train_samples_per_second': 0.479, 'train_steps_per_second': 0.03, 'total_flos': 1.4273634039005184e+16, 'train_loss': 0.9570221038574868, 'entropy': 0.8451851050059, 'num_tokens': 267036.0, 'mean_token_accuracy': 0.8022247771422069, 'epoch': 1.0})

# Save the model in Drive

In [11]:
adapter_path = f"{SAVE_DIR}/final_adapter"
trainer.model.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)
print(f"Saved to {adapter_path}")

Saved to /content/drive/MyDrive/qlora-sql-project/final_adapter


# Load a test split from the dataset

In [12]:
eval_dataset = load_dataset("b-mc2/sql-create-context", split="train")
eval_dataset = eval_dataset.shuffle(seed=42).select(range(3000, 3150))
print(f"Holdout size: {len(eval_dataset)}")

Holdout size: 150


# Generate SQL, run an output checker and create statistics on how many outputs were on point

In [17]:
import sqlite3
import re

def generate_sql(model, tokenizer, context, question):
    prompt = f"""[INST] Given this database schema:
{context}

Write a SQL query for: {question} [/INST]"""
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    input_length = inputs['input_ids'].shape[1]
    output = model.generate(**inputs, max_new_tokens=150, do_sample=False)
    generated_tokens = output[0][input_length:]
    generated_sql = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()
    return generated_sql

def check_correct(context, generated_sql, gold_sql):
    try:
        conn = sqlite3.connect(":memory:")
        cur = conn.cursor()
        cur.execute(context)
        cur.execute(generated_sql)
        generated_result = set(cur.fetchall())
        cur.execute(gold_sql)
        gold_result = set(cur.fetchall())
        conn.close()
        return generated_result == gold_result
    except Exception:
        return False

def run_eval(model, tokenizer, eval_dataset):
    correct = 0
    for example in eval_dataset:
        sql = generate_sql(model, tokenizer, example['context'], example['question'])
        if check_correct(example['context'], sql, example['answer']):
            correct += 1
    accuracy = correct / len(eval_dataset)
    return accuracy

# Sample to test fine-tuned model

In [18]:
example = eval_dataset[0]
sql = generate_sql(model, tokenizer, example['context'], example['question'])
print("Generated:", sql)
print("Gold:", example['answer'])
print("Correct:", check_correct(example['context'], sql, example['answer']))

Generated: SELECT tries_for FROM table_12828723_4 WHERE points_for = "473"
Gold: SELECT COUNT(tries_for) FROM table_12828723_4 WHERE points_for = "473"
Correct: False


# Eval fine-tuned model

In [19]:
finetuned_accuracy = run_eval(model, tokenizer, eval_dataset)
print(f"Fine-tuned model accuracy: {finetuned_accuracy:.2%}")

Fine-tuned model accuracy: 94.00%


# Load base model

In [20]:
from transformers import AutoModelForCausalLM

base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
)
base_model.eval()

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

MistralForCausalLM(
  (model): MistralModel(
    (embed_tokens): Embedding(32768, 4096)
    (layers): ModuleList(
      (0-31): 32 x MistralDecoderLayer(
        (self_attn): MistralAttention(
          (q_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
        )
        (mlp): MistralMLP(
          (gate_proj): Linear4bit(in_features=4096, out_features=14336, bias=False)
          (up_proj): Linear4bit(in_features=4096, out_features=14336, bias=False)
          (down_proj): Linear4bit(in_features=14336, out_features=4096, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): MistralRMSNorm((4096,), eps=1e-05)
        (post_attention_layernorm): MistralRMSNorm((4096,), eps=1e-05)
      )
    )
    (n

# Sample to test base model

In [22]:
example = eval_dataset[0]
sql = generate_sql(base_model, tokenizer, example['context'], example['question'])
print("Base model generated:", sql)

Base model generated: To write a SQL query for the given schema, we need to find the number of rows where the `tries_for` column is equal to a specific value (let's assume it's 'x') and the `points_for` column is equal to '473'. Here's the SQL query:

```sql
SELECT COUNT(*)
FROM table_12828723_4
WHERE tries_for = 'x' AND points_for = '473';
```

Replace 'x' with the actual value of the `tries_for` column you want to search for. This query will return the count of rows where both


# Eval base model

In [21]:
base_accuracy = run_eval(base_model, tokenizer, eval_dataset)
print(f"Base model accuracy: {base_accuracy:.2%}")

Base model accuracy: 0.00%
